## LangChain: Q&A over Documents

An example might be a tool that would allow you to query a product catalog for items of interest.

In [38]:
# Import required packages
import os
from langchain_classic.chains import RetrievalQA
from langchain_classic.document_loaders import CSVLoader
from langchain_classic.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown

from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any
import requests

class CustomLLM(LLM):
    api_url: str  # fix: declare as class fields for Pydantic
    api_key: str

    @property
    def _llm_type(self) -> str:
        return "custom"
        
    def get_num_tokens(self, text: str) -> int:
        # rough estimate: ~4 chars per token
        return len(text) // 4
        
    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager: Any = None) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": "your-model-name",  # may be required by your server
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.7,
            "max_tokens": 100
        }
        response = requests.post(self.api_url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"]  # fix: correct response path

    @property
    def _identifying_params(self) -> dict:
        return {
            "api_url": self.api_url,
            "api_key": self.api_key
        }

# Use the custom LLM
custom_llm = CustomLLM(api_url="http://127.0.0.1:8000/v1/chat/completions", api_key="chenyang216")

# fix: use __call__ or invoke(), not .run()
response = custom_llm.invoke("What is the capital of France?")
print(response)

**Paris** is the **capital of France**. 

Paris sits in north‑central France along the Seine River and serves as the country’s political, cultural, and economic center. 



If you want to explore more, you can dive into **[Paris](ca://s?q=Tell_me_more_about_Paris)** or compare **[French cities](ca://s?q=Compare_major_French_cities)**.


In [4]:
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding='utf-8')

In [5]:
from langchain_classic.indexes import VectorstoreIndexCreator

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",  # small, fast, good quality
    model_kwargs={"device": "cpu"},  # or "cuda" if you have GPU
    encode_kwargs={"normalize_embeddings": True},
)

index = VectorstoreIndexCreator(
    embedding=embedding,
    vectorstore_cls=DocArrayInMemorySearch
).from_loaders([loader])

C:\Users\HW\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HW\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6769.78it/s]


In [8]:
query ="Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

In [10]:
response = index.query(query,llm=custom_llm)
display(Markdown(response))

All four items you provided are sun‑protective shirts (each rated **UPF 50+**).  
Here’s a clean, structured markdown table followed by concise summaries of each one.

---

### ☀️ Sun‑Protection Shirts Overview

| **Shirt** | **Fit** | **Fabric** | **UPF Rating** | **Key Features** |
|-----------|---------|------------|----------------|------------------|
| **[Women’s Tropical Tee, Sleeveless](ca://s?q=Tell_me_more_about_Women%E2%80%99s_Tropical_Tee_Sleeveless)** | Slightly fitted | 71% nylon / 29% polyester | UPF 50+ | Wrinkle‑resistant, vented cape design, low‑profile pockets, SunSmart™ fabric |
| **[Sun Shield Shirt](ca://s?q=Tell_me_more_about_Sun_Shield_Shirt)** | Slightly fitted | 78% nylon / 22% Lycra Xtra Life | UPF 50+ | Moisture‑wicking, abrasion‑resistant, swimsuit‑friendly, Skin Cancer Foundation recommended |
| **[Tropical Breeze Shirt](ca://s?q=Tell_me_more_about_Tropical_Breeze_Shirt)** | Traditional fit | 71% nylon / 29% polyester | UPF 50+ | Lightweight, breathable, moisture‑wicking, wrinkle‑free, designed for hot weather |
| **[Men’s Plaid Tropic Shirt, Short‑Sleeve](ca://s?q=Tell_me_more_about_Men%E2%80%99s_Plaid_Tropic_Shirt_Short_Sleeve)** | Not specified (relaxed style implied) | 52% polyester / 48% nylon | UPF 50+ | Cape venting, bellows pockets, wrinkle‑free, quick‑drying, fishing/travel design |

---

### 🌴 Summaries

#### **Women’s Tropical Tee, Sleeveless**
A lightweight sleeveless button‑up designed to flatter while providing **SunSmart™ UPF 50+** protection. Wrinkle‑resistant, vented, and equipped with subtle pockets for a clean look. Great for warm‑weather activity with durable sun protection built into the fabric.



---

#### **Sun Shield Shirt**
A high‑performance sun shirt made with Lycra Xtra Life fiber for durability. Offers **UPF 50+**, moisture‑wicking comfort, and abrasion resistance. Designed to layer over swimwear and recommended by The Skin Cancer Foundation for UV protection.



---

#### **Tropical Breeze Shirt**
A men’s long‑sleeve hot‑weather shirt with **SunSmart™ UPF 50+** coverage. Lightweight, breathable, wrinkle‑free, and quick‑drying—originally built for fishing but ideal for travel and extended outdoor use.



---

#### **Men’s Plaid Tropic Shirt, Short‑Sleeve**
A short‑sleeve men’s shirt offering **UPF 50+** protection with quick‑drying, wrinkle‑free fabric. Includes cape venting and bellows pockets, making it a practical choice for fishing, travel, and hot climates.



---

If you want, I can also create a **comparison table**, a **recommendation based on activity**, or a **buyer’s guide**—just choose one:  
[comparison](ca://s?q=Create_a_comparison_table), [activity_recommendation](ca://s?q=Recommend_shirts_based_on_activity), or [buyer%27s_guide](ca://s?q=Create_a_sun_shirt_buyer%27s_guide).

In [34]:
loader = CSVLoader(file_path=file, encoding='utf-8')
docs = loader.load()
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [14]:
embed = embedding.embed_query("Hi my name is Harrison")
print(len(embed))
print(embed[:5])

384
[-0.06155332550406456, -0.06207890808582306, -0.018953032791614532, 0.048299096524715424, -0.028553972020745277]


In [16]:
db = DocArrayInMemorySearch.from_documents(
    docs, 
    embedding
)

In [17]:
query = "Please suggest a shirt with sunblocking"
docs = db.similarity_search(query)

In [37]:
len(docs)

1000

In [19]:
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 255}, page_content=': 255\nname: Sun Shield Shirt by\ndescription: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. \n\nSize & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.\n\nFabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.\n\nAdditional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.\n\nSun Protection That Won\'t Wear Off\nOur high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun\'s harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.')

In [22]:
retriever = db.as_retriever()

In [23]:
qdocs = "".join([docs[i].page_content for i in range(len(docs))])

In [25]:
response = custom_llm.invoke(f"{qdocs} Question: Please list all your \
shirts with sun protection in a table in markdown and summarize each one.")

display(Markdown(response))

All four shirts you provided offer **UPF 50+ sun protection**, so here is a clean, structured **markdown table** followed by **clear summaries** of each one.

---

### ☀️ Sun‑Protection Shirts Overview  
All items below feature **UPF 50+**, blocking **98% of harmful UV rays**.

| **Shirt Name** | **Fit** | **Fabric** | **Key Features** | **Summary** |
|----------------|---------|------------|------------------|-------------|
| **[Sun Shield Shirt](ca://s?q=Tell_me_more_about_Sun_Shield_Shirt)** | Slightly Fitted | 78% nylon, 22% Lycra Xtra Life | Quick‑drying, abrasion‑resistant, swimsuit‑friendly | High‑performance sun shirt with moisture‑wicking comfort and durable stretch fabric. |
| **[Tropical Breeze Shirt](ca://s?q=Tell_me_more_about_Tropical_Breeze_Shirt)** | Traditional Fit | 71% nylon, 29% polyester + mesh | Wrinkle‑resistant, cape venting, travel‑ready | Lightweight long‑sleeve shirt designed for hot weather, fishing, and extended travel. |
| **[Men’s Plaid Tropic Shirt](ca://s?q=Tell_me_more_about_Men%27s_Plaid_Tropic_Shirt)** | Traditional Fit | 52% polyester, 48% nylon | Cape venting, quick‑evaporation, travel‑friendly | Short‑sleeve hot‑weather shirt with fast‑drying fabric and breathable venting. |
| **[Women’s Tropical Tee, Sleeveless](ca://s?q=Tell_me_more_about_Women%27s_Tropical_Tee_Sleeveless)** | Slightly Fitted | 71% nylon, 29% polyester | Wrinkle‑resistant, pockets, eyewear loop | Sleeveless UPF shirt with flattering fit, smooth buttons, and practical outdoor features. |

---

## 🌤️ Summaries of Each Shirt

### **[Sun Shield Shirt](ca://s?q=Tell_me_more_about_Sun_Shield_Shirt)**  
A performance‑driven sun shirt built for active use. Its nylon–Lycra blend gives stretch, durability, and excellent moisture‑wicking. Designed to fit comfortably over swimwear, it’s abrasion‑resistant and ideal for long days outdoors.

---

### **[Tropical Breeze Shirt](ca://s?q=Tell_me_more_about_Tropical_Breeze_Shirt)**  
A lightweight, breathable long‑sleeve shirt originally made for fishing but perfect for any hot‑weather activity. Cape venting and mesh inserts keep air flowing, while wrinkle‑free fabric makes it great for travel. UPF 50+ ensures maximum sun protection.

---

### **[Men’s Plaid Tropic Shirt, Short‑Sleeve](ca://s?q=Tell_me_more_about_Men%27s_Plaid_Tropic_Shirt)**  
A short‑sleeve option for hot climates, combining quick‑evaporating fabric with UPF 50+ coverage. Cape venting and bellows pockets add practicality, making it a reliable shirt for outdoor work, travel, or fishing.

---

### **[Women’s Tropical Tee, Sleeveless](ca://s?q=Tell_me_more_about_Women%27s_Tropical_Tee_Sleeveless)**  
A sleeveless button‑up with a flattering slightly‑fitted silhouette. It includes low‑profile pockets, smoother updated buttons, and cape venting for airflow. Lightweight, wrinkle‑resistant, and built with SunSmart UPF 50+ protection.

---

If you want, I can also create **a comparison table**, **rank them by breathability**, or **recommend the best one for travel, fishing, or everyday wear** — just choose one: [compare them](ca://s?q=Compare_all_sun_shirts), [rank by breathability](ca://s?q=Rank_sun_shirts_by_breathability), or [recommend one](ca://s?q=Recommend_best_sun_shirt).

In [26]:
qa_stuff = RetrievalQA.from_chain_type(
    llm=custom_llm, 
    chain_type="stuff", 
    retriever=retriever, 
    verbose=True
)

In [27]:
query =  "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

In [28]:
response = qa_stuff.run(query)
display(Markdown(response))

C:\Users\HW\AppData\Local\Temp\ipykernel_8060\168711354.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = qa_stuff.run(query)




> Entering new RetrievalQA chain...

> Finished chain.


All four items you provided include **built‑in UPF 50+ sun protection**, so they all qualify.  
Here is a clear, structured table followed by concise summaries of each shirt.

---

### ☀️ Sun‑Protective Shirts (Markdown Table)

| **Shirt** | Fit | Fabric | Key Sun‑Protection Features |
|-----------|------|---------|-----------------------------|
| **[Women’s Tropical Tee, Sleeveless](ca://s?q=Tell_me_more_about_Women’s_Tropical_Tee_Sleeveless)** | Slightly Fitted | 71% nylon, 29% polyester | SunSmart™ UPF 50+, blocks 98% UV rays, vented cape design |
| **[Sun Shield Shirt](ca://s?q=Tell_me_more_about_Sun_Shield_Shirt)** | Slightly Fitted | 78% nylon, 22% Lycra Xtra Life | UPF 50+, Skin Cancer Foundation–recommended fabric, moisture‑wicking |
| **[Sunrise Tee](ca://s?q=Tell_me_more_about_Sunrise_Tee)** | Slightly Fitted | 71% nylon, 29% polyester | SunSmart™ UPF 50+, wrinkle‑free, fast‑drying, vented cape |
| **[Tropical Breeze Shirt](ca://s?q=Tell_me_more_about_Tropical_Breeze_Shirt)** | Traditional Fit | 71% nylon, 29% polyester | UPF 50+, breathable mesh inserts, moisture‑wicking |

---



---

### 🌴 Summaries of Each Sun‑Protective Shirt

- **[Women’s Tropical Tee, Sleeveless](ca://s?q=Tell_me_more_about_Women’s_Tropical_Tee_Sleeveless)** — A lightweight sleeveless button‑up with SunSmart™ UPF 50+ protection, flattering shaping, wrinkle resistance, and vented cape panels for airflow.

- **[Sun Shield Shirt](ca://s?q=Tell_me_more_about_Sun_Shield_Shirt)** — A high‑performance sun shirt made with abrasion‑resistant, moisture‑wicking fabric. UPF 50+ rated and recommended by the Skin Cancer Foundation for reliable UV protection.

- **[Sunrise Tee](ca://s?q=Tell_me_more_about_Sunrise_Tee)** — A women’s hot‑weather button‑down designed for fishing and travel. It dries quickly, resists wrinkles, and includes SunSmart™ UPF 50+ protection with venting for comfort.

- **[Tropical Breeze Shirt](ca://s?q=Tell_me_more_about_Tropical_Breeze_Shirt)** — A men’s breathable long‑sleeve UPF 50+ shirt with mesh inserts, moisture‑wicking fabric, and relaxed fit. Ideal for extended outdoor use and travel.

---

If you want, I can also create a **comparison table**, a **buyer’s guide**, or help you **categorize these shirts by gender, activity, or fabric**.

In [30]:
response = index.query(query, llm=custom_llm)

In [31]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embedding,
).from_loaders([loader])